# 02. 전처리 & 데이터셋 구성

`01_EDA`에서 얻은 결론을 반영해 모델링용 데이터를 만든다.

### EDA에서 넘어온 결정사항
1. **결과 파생 컬럼 제거** — `Time`, `Event`, `Harvest`, `Alive`는 결과를 알고 난 뒤 기록되는 값이라 분류 피처로 쓰면 누수. 제외한다.
2. **식별자 컬럼 제거** — `No, Plot, Subplot, Core, Census, Adult`는 실험 관리용.
3. **결측치** — `EMF`는 0으로, `Event` 결측 1행은 제거.
4. **Phenolics 음수 보정** — 최솟값을 0으로 이동.
5. **수확(Harvested) 개체 제외** — 실험자 개입이라 자연 생존/사망 분석에서 뺀다.

### 두 개의 데이터셋을 만든다
- **분류용** (`03_classification`): 관측 종료 시점 기준 *생존 vs 사망* 이진 분류. 누수 컬럼 전부 제외.
- **생존분석용** (`04_survival_analysis`): *시간에 따른* 생존을 예측. 여기서는 `Time`이 예측 대상(생존 시간)이므로 정당하게 사용한다.

In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv("../data/raw/Tree_Data.csv")
print("원본:", df.shape)

원본: (2783, 24)


## 1. 라벨 통합 & 기본 정제

In [2]:
d = df.copy()

# 세 플래그 -> 하나의 Status (0=사망, 1=수확, 2=생존)
d["Status"] = np.nan
d.loc[d["Event"] == 1, "Status"] = 0
d.loc[d["Harvest"] == "X", "Status"] = 1
d.loc[d["Alive"] == "X", "Status"] = 2
d = d.dropna(subset=["Status"]).copy()      # Event 결측 1행 제거
d["Status"] = d["Status"].astype(int)

# 결측치 처리: EMF 미검출 -> 0
d["EMF"] = d["EMF"].fillna(0)

# Phenolics 음수 보정 (측정 특성상 극소 함유량에서 음수 발생 -> 최솟값을 0으로)
d["Phenolics"] = d["Phenolics"] - d["Phenolics"].min()

print("정제 후:", d.shape)
print("Phenolics 최솟값:", d["Phenolics"].min())
print("결측치 남은 컬럼:", d.isna().sum()[d.isna().sum() > 0].index.tolist())

정제 후: (2782, 25)
Phenolics 최솟값: 0.0
결측치 남은 컬럼: ['Harvest', 'Alive']


## 2. 컬럼 분류

- **식별자**(제거): `No, Plot, Subplot, Core, Census, Adult`
- **결과 파생/누수**(분류에서 제거): `Event, Harvest, Alive, Time, PlantDate`
- **피처 후보**: 수종·환경·균근·화학 성분

In [3]:
id_cols   = ["No", "Plot", "Subplot", "Core", "Census", "Adult"]
leak_cols = ["Event", "Harvest", "Alive", "Time", "PlantDate"]

feature_cols = [c for c in d.columns
                if c not in id_cols + leak_cols + ["Status"]]
print("피처 후보:", feature_cols)

피처 후보: ['Species', 'Light_ISF', 'Light_Cat', 'Soil', 'Sterile', 'Conspecific', 'Myco', 'SoilMyco', 'AMF', 'EMF', 'Phenolics', 'Lignin', 'NSC']


## 3. 범주형 인코딩

범주형 7개를 Label Encoding한다. 트리 기반 모델(RandomForest, XGBoost)이 주력이라 label encoding으로 충분하고, 인코딩 맵은 나중에 해석을 위해 보관한다.

In [4]:
cat_cols = d[feature_cols].select_dtypes(include=["object"]).columns.tolist()
print("범주형:", cat_cols)

encoders = {}
d_enc = d.copy()
for c in cat_cols:
    cats = d_enc[c].astype("category")
    encoders[c] = dict(enumerate(cats.cat.categories))   # {코드: 원본값}
    d_enc[c] = cats.cat.codes

for c in cat_cols:
    print(f"  {c}: {encoders[c]}")

범주형: ['Species', 'Light_Cat', 'Soil', 'Sterile', 'Conspecific', 'Myco', 'SoilMyco']
  Species: {0: 'Acer saccharum', 1: 'Prunus serotina', 2: 'Quercus alba', 3: 'Quercus rubra'}
  Light_Cat: {0: 'High', 1: 'Low', 2: 'Med'}
  Soil: {0: 'Acer rubrum', 1: 'Acer saccharum', 2: 'Populus grandidentata', 3: 'Prunus serotina', 4: 'Quercus alba', 5: 'Quercus rubra', 6: 'Sterile'}
  Sterile: {0: 'Non-Sterile', 1: 'Sterile'}
  Conspecific: {0: 'Conspecific', 1: 'Heterospecific', 2: 'Sterilized'}
  Myco: {0: 'AMF', 1: 'EMF'}
  SoilMyco: {0: 'AMF', 1: 'EMF', 2: 'Sterile'}


## 4. 분류용 데이터셋

- 수확(Status=1) 제외 → 생존(2) vs 사망(0)
- 타깃 `Survived`: 생존=1, 사망=0
- 누수·식별자 컬럼 전부 제외

In [5]:
clf = d_enc[d_enc["Status"].isin([0, 2])].copy()
clf["Survived"] = (clf["Status"] == 2).astype(int)
clf = clf[feature_cols + ["Survived"]]

print("분류용:", clf.shape)
print("생존율:", clf["Survived"].mean().round(3))
clf.head()

분류용: (2078, 14)
생존율: 0.236


,Species,Light_ISF,Light_Cat,Soil,Sterile,Conspecific,Myco,SoilMyco,AMF,EMF,Phenolics,Lignin,NSC,Survived
0,0,0.106,2,3,0,1,0,0,22.00,0.00,0.79,13.86,12.15,0
1,2,0.106,2,5,0,1,1,1,15.82,31.07,6.54,20.52,19.29,1
2,3,0.106,2,3,0,1,1,0,24.45,28.19,4.71,24.74,15.01,0
3,0,0.080,2,3,0,1,0,0,22.23,0.00,0.64,14.29,12.36,0
4,0,0.060,1,3,0,1,0,0,21.15,0.00,0.77,10.85,11.20,0


### 재현 가능한 데이터 분할

`train_test_split`에 `random_state`를 고정한다. (팀 원본은 seed가 없어 실행마다 결과가 바뀌었다.) 클래스 비율 유지를 위해 `stratify` 적용.

In [6]:
from sklearn.model_selection import train_test_split

X = clf.drop(columns="Survived")
y = clf["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"train {X_train.shape}, test {X_test.shape}")
print(f"train 생존율 {y_train.mean():.3f}, test 생존율 {y_test.mean():.3f}")

train (1662, 13), test (416, 13)
train 생존율 0.236, test 생존율 0.236


## 5. 생존분석용 데이터셋

시간에 따른 생존을 다루므로 `Time`을 유지한다.
- `duration` = `Time` (사건 발생까지의 시간)
- `event` = 사망 여부 (사망=1, 수확·생존=0 → **censored**)

수확·생존 개체는 "관측 시점까지 사망하지 않았다"는 정보(중도절단, censored)로 활용된다. 이것이 생존분석이 단순 분류보다 데이터를 더 잘 쓰는 지점이다.

In [7]:
surv = d_enc.copy()
surv["event"] = (surv["Status"] == 0).astype(int)   # 사망=1, censored=0
surv["duration"] = surv["Time"]
surv = surv[feature_cols + ["duration", "event"]]

print("생존분석용:", surv.shape)
print("event(사망=1):", surv["event"].value_counts().to_dict())
print("duration:", surv["duration"].min(), "~", surv["duration"].max())
surv.head()

생존분석용: (2782, 15)
event(사망=1): {1: 1587, 0: 1195}
duration: 14.0 ~ 115.5


,Species,Light_ISF,Light_Cat,Soil,Sterile,Conspecific,Myco,SoilMyco,AMF,EMF,Phenolics,Lignin,NSC,duration,event
0,0,0.106,2,3,0,1,0,0,22.00,0.00,0.79,13.86,12.15,14.0,1
1,2,0.106,2,5,0,1,1,1,15.82,31.07,6.54,20.52,19.29,115.5,0
2,3,0.106,2,3,0,1,1,0,24.45,28.19,4.71,24.74,15.01,63.0,1
3,0,0.080,2,3,0,1,0,0,22.23,0.00,0.64,14.29,12.36,14.0,1
4,0,0.060,1,3,0,1,0,0,21.15,0.00,0.77,10.85,11.20,14.0,1


## 6. 저장

`data/processed/`에 저장한다. (이 파일들은 노트북 실행으로 재생성되므로 `.gitignore` 대상으로 둘 수 있다.)

In [8]:
import os
os.makedirs("../data/processed", exist_ok=True)

clf.to_csv("../data/processed/clf_dataset.csv", index=False)
surv.to_csv("../data/processed/surv_dataset.csv", index=False)

# 인코딩 맵도 보관
import json
with open("../data/processed/encoders.json", "w", encoding="utf-8") as f:
    json.dump({k: {str(kk): vv for kk, vv in v.items()} for k, v in encoders.items()},
              f, ensure_ascii=False, indent=2)

print("저장 완료:")
print("  - clf_dataset.csv ", clf.shape)
print("  - surv_dataset.csv", surv.shape)
print("  - encoders.json")

저장 완료:
  - clf_dataset.csv  (2078, 14)
  - surv_dataset.csv (2782, 15)
  - encoders.json


## 요약

| 산출물 | 용도 | 형태 | 특징 |
|--------|------|------|------|
| `clf_dataset.csv` | 03 분류 | 2,078 × 14 | 수확 제외, 누수 컬럼 제거, 생존/사망 이진 |
| `surv_dataset.csv` | 04 생존분석 | 2,782 × 15 | Time=duration 유지, censored 포함 |
| `encoders.json` | 해석용 | - | 범주형 label ↔ 원본값 매핑 |

다음: `03_classification`에서 누수를 제거한 정직한 분류 성능을 확인한다.